# Ch. 4 · Réseaux neuronaux MLP

**IA pour les sciences — M1.** Comparer un MLP à un modèle plus simple et contrôler la généralisation.

Exécutez les cellules dans l’ordre. La démarche et ses limites sont discutées dans le chapitre HTML correspondant. Les données artificielles utilisées, le cas échéant, servent uniquement à l’enseignement.

## Préparation de l’environnement

Dépendances usuelles : `numpy`, `pandas`, `matplotlib`, `scikit-learn` (et `scipy` pour le chapitre 6). Sur Colab, elles sont généralement disponibles.

### Étape 1

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split

iris = load_iris(as_frame=True)
X, y = iris.data, iris.target
print(X.shape, iris.target_names)
print(X.describe().round(2))
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=42)

### Étape 2

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

baseline = make_pipeline(
    StandardScaler(), LogisticRegression(max_iter=1000))
baseline.fit(X_train, y_train)
base_pred = baseline.predict(X_test)
print('Baseline accuracy:', accuracy_score(y_test, base_pred))
print(classification_report(y_test, base_pred,
                            target_names=iris.target_names))

### Étape 3

In [ ]:
from sklearn.neural_network import MLPClassifier

mlp = make_pipeline(
    StandardScaler(),
    MLPClassifier(hidden_layer_sizes=(16, 8),
                  activation='relu', solver='adam',
                  alpha=0.01, max_iter=1500,
                  early_stopping=True,
                  validation_fraction=0.2,
                  random_state=42))
mlp.fit(X_train, y_train)
mlp_pred = mlp.predict(X_test)
print('MLP accuracy:', accuracy_score(y_test, mlp_pred))
print(classification_report(y_test, mlp_pred,
                            target_names=iris.target_names))

network = mlp.named_steps['mlpclassifier']
plt.plot(network.loss_curve_)
plt.xlabel('Itération'); plt.ylabel('Loss entraînement')
plt.title('Historique de perte du MLP')
plt.show()

### Étape 4

In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_val_score

cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
for layers in [(8,), (16, 8), (32, 16)]:
    pipe = make_pipeline(
        StandardScaler(),
        MLPClassifier(hidden_layer_sizes=layers,
                      max_iter=1500, alpha=0.01,
                      random_state=42))
    scores = cross_val_score(pipe, X_train, y_train,
                             cv=cv, scoring='accuracy')
    print(layers, 'accuracy CV:', round(scores.mean(), 3),
          '+/-', round(scores.std(), 3))

## Interprétation scientifique

1. Décrivez les données et leurs limites.
2. Justifiez la méthode et les métriques.
3. Discutez ce que les résultats permettent de conclure et ce qu’ils **ne permettent pas** de conclure.
4. Proposez au moins un contrôle de robustesse ou une expérience complémentaire.